# 3.0 — Retirement (DNF) prediction dataset

Building a dataset for the question: **will this driver finish this race?**

Three things make this different from the points model in notebook 2.0:

1. **The target is rare and the metric matters.** Roughly one car in seven retires, so a
   model predicting "everyone finishes" is ~86% accurate and carries no information.
   Nothing here reports accuracy. The headline number is **Brier skill** — how much the
   forecast improves on always predicting the base rate.
2. **The circuit is measured, not named.** Instead of a one-hot per venue, every circuit
   is characterised from position and speed telemetry: curvature, lateral load, full-throttle
   share, braking-zone density. That transfers to a circuit the model has never seen.
3. **Leakage is enforced, not assumed.** Every history feature is shifted before it is
   aggregated, and a detector flips one race's outcomes and rebuilds to prove nothing moved.

Everything below imports from `src/`, so the same code runs in the notebook, in a script
and in the tests. No hard-coded paths.


In [ ]:
import sys, warnings, logging
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(message)s")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from src import config
from src.features import registry
print("repo root:", config.REPO_ROOT)
print("seasons  :", config.DEFAULT_SEASONS)

## 1. Build the dataset

The pull needs network access to `livetiming.formula1.com` and `api.jolpi.ca`. Check first —
some networks refuse both, and the error is much clearer up front than three retries deep.

In [ ]:
from src.data import ingest

ingest.configure()
reachable, message = ingest.check_connectivity()
print(message)

In [ ]:
# Full build. The circuit profiles are the slow part: one telemetry session per event.
# Expect ~30-60 min cold, seconds warm, and a few GB of cache.
#
#     python -m src.data.generate_dataset --seasons 2018-2025
#
# Re-run feature logic without touching the network:
#
#     python -m src.data.generate_dataset --skip-download

REBUILD = False  # flip to True to run the download here

if REBUILD and reachable:
    results, profiles = ingest.collect_results(config.DEFAULT_SEASONS)
    print(results.summary() if hasattr(results, "summary") else f"{len(results)} rows")
    profiles, profile_report = ingest.collect_circuit_profiles(config.DEFAULT_SEASONS)
    print(profile_report.summary())
    config.ensure_dirs()
    results.to_parquet(config.RACE_RESULTS_PATH, index=False)
    profiles.to_parquet(config.CIRCUIT_PROFILE_PATH, index=False)

In [ ]:
from src.data.generate_dataset import build_dataset

if config.RACE_RESULTS_PATH.exists():
    raw_results = pd.read_parquet(config.RACE_RESULTS_PATH)
    profiles = (pd.read_parquet(config.CIRCUIT_PROFILE_PATH)
                if config.CIRCUIT_PROFILE_PATH.exists() else None)
    SOURCE = "real"
else:
    # No cache yet: fall back to the synthetic fixture so the notebook still runs.
    # It validates the plumbing only — never read a finding off synthetic data.
    print("No cached results found; using the synthetic fixture.")
    from tests.synthetic import (make_race_results, make_circuit_telemetry,
                                 make_fourier_telemetry, monza_like, monaco_like,
                                 FAST_FOURIER_COEFFS, TWISTY_FOURIER_COEFFS)
    from src.features.track_profile import build_lap_profile

    raw_results = make_race_results(seasons=tuple(range(2018, 2025)),
                                    races_per_season=20, n_drivers=20, n_circuits=5, seed=11)
    raw_results["session_type"] = "R"
    builders = {100: lambda: make_circuit_telemetry(monza_like(), step_m=1.0),
                101: lambda: make_circuit_telemetry(monaco_like(), step_m=1.0),
                102: lambda: make_fourier_telemetry(FAST_FOURIER_COEFFS, base_radius_m=900),
                103: lambda: make_fourier_telemetry(TWISTY_FOURIER_COEFFS, base_radius_m=600),
                104: lambda: make_fourier_telemetry(FAST_FOURIER_COEFFS, base_radius_m=700)}
    profiles = pd.DataFrame([
        build_lap_profile(fn(), metadata={"circuit_key": k, "circuit_name": str(k), "year": y})
        for k, fn in builders.items() for y in range(2018, 2025)])
    SOURCE = "synthetic"

dataset, diagnostics = build_dataset(raw_results, profiles)
print(f"\nsource={SOURCE}  rows={len(dataset)}  columns={dataset.shape[1]}  "
      f"dnf rate={dataset['dnf'].mean():.3f}")

## 2. Check the labels before anything else

If the target is wrong, no amount of model tuning will reveal it. Two cases decide whether
this is right: a driver who retires past 90% distance (still classified, but did not finish),
and a driver disqualified after taking the flag (classified out, but did finish).

In [ ]:
print(diagnostics["label_summary"].to_string(index=False))
print()
print("Retired but still classified (the >90% distance case):",
      int(dataset["dnf_classified"].sum()))
print("Strict DNFs (retired and unclassified):", int(dataset["dnf_strict"].sum()))

In [ ]:
# Any status the taxonomy did not recognise falls into `other`. A real cause appearing
# here means _CAUSE_RULES needs extending — it should never be silently absorbed.
from src.features.labels import unmapped_statuses
leftovers = unmapped_statuses(raw_results["Status"])
print(leftovers.to_string() if not leftovers.empty else "every status recognised")

## 3. The leakage check

This is the part worth being pedantic about. The detector flips one event's outcomes,
rebuilds the entire feature table, and reports any feature that moved at or before that
event. A clean result means no feature can see the race it is predicting.

It is tested against deliberately planted leaks, so a pass means something. It has already
caught two real bugs in this pipeline: a team aggregate joined back from the current race,
and a `(Year, RoundNumber)` key that stopped being unique on sprint weekends.

In [ ]:
leakage = diagnostics["leakage"]
print("PASS — no feature reads the current race" if leakage.empty
      else leakage.to_string(index=False))

In [ ]:
# Confirm the detector is capable of firing. If this cell prints nothing, the check above
# is worthless.
from src.features.build_features import build_history_features, detect_target_leakage
from src.features.labels import add_race_outcome_labels

started = add_race_outcome_labels(raw_results, warn_on_unmapped=False)
started = started.loc[started["started"] == 1].reset_index(drop=True)

def leaky(frame):
    out = build_history_features(frame)
    out["planted_leak"] = out.sort_values("RaceDate").groupby("DriverId")["dnf"].cumsum()
    return out

print(detect_target_leakage(started, builder=leaky).to_string(index=False))

## 4. What the circuits actually look like

This is the feature block you asked for. Every number below is measured from a reference
lap's position and speed telemetry, not looked up.

* `track_speed_index` — z-scored composite of mean speed, high-speed share and
  full-throttle share. Monza high, Monaco low.
* `mechanical_stress_index` — full throttle, braking density, peak deceleration, gear
  changes. The hypothesis for *car* failures.
* `incident_exposure_index` — corner density, low-speed share, corner radius. The
  hypothesis for *driver* failures.

The two stress indices are hypotheses, not findings. Section 7 tests them.

In [ ]:
track_cols = ["track_speed_index", "speed_mean_kph", "pct_dist_above_250kph",
              "pct_full_throttle", "corners_per_km", "median_corner_radius_m",
              "braking_zones_per_km", "lat_g_mean", "elevation_range_m",
              "mechanical_stress_index", "incident_exposure_index"]
have = [c for c in track_cols if c in dataset.columns]

name_col = "EventName" if "EventName" in dataset.columns else "circuit_key"
by_circuit = (dataset.groupby(name_col, observed=True)[have]
              .mean()
              .sort_values("track_speed_index", ascending=False))
by_circuit.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ranked = by_circuit.sort_values("track_speed_index")
colors = plt.cm.RdYlBu(np.linspace(0.1, 0.9, len(ranked)))
ax.barh(range(len(ranked)), ranked["track_speed_index"], color=colors)
ax.set_yticks(range(len(ranked)))
ax.set_yticklabels(ranked.index, fontsize=8)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("track speed index  (standard deviations from the average circuit)")
ax.set_title("Circuits ranked by measured speed character")
plt.tight_layout()

In [ ]:
# Does the circuit's character line up with how often cars actually retire there?
# Note the sample: eight seasons is roughly eight visits per circuit, so these are
# noisy. Read the direction, not the magnitude.
observed = (dataset.groupby(name_col, observed=True)
            .agg(dnf_rate=("dnf", "mean"),
                 mech_rate=("dnf_mechanical", "mean"),
                 incident_rate=("dnf_incident", "mean"),
                 races=("dnf", "size"),
                 speed=("track_speed_index", "mean"),
                 mech_stress=("mechanical_stress_index", "mean"),
                 incident_exposure=("incident_exposure_index", "mean")))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (x, y, title) in zip(axes, [
        ("mech_stress", "mech_rate", "mechanical stress vs mechanical retirements"),
        ("incident_exposure", "incident_rate", "incident exposure vs incident retirements")]):
    ax.scatter(observed[x], observed[y], s=observed["races"] * 3, alpha=0.7)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(title)
    if observed[[x, y]].dropna().shape[0] > 2:
        r = observed[[x, y]].corr(method="spearman").iloc[0, 1]
        ax.annotate(f"spearman = {r:+.2f}", xy=(0.05, 0.92), xycoords="axes fraction")
plt.tight_layout()

## 5. Which features may a model use, and when?

Grid position is one of the strongest single predictors of retirement — the back of the
grid both breaks more and gets collected at turn one more. But it is unknown until Saturday.
The registry tags every feature so that choice is made once, in code, rather than by hand
each time.

In [ ]:
frame = registry.registry_frame()
print(frame.groupby(["stage", "kind"]).size().to_string())
print()
for stage in registry.STAGE_ORDER:
    usable = registry.feature_columns(stage, available=dataset.columns)
    print(f"{stage:12s}: {len(usable):3d} features present in this dataset")

In [ ]:
audit = diagnostics["registry_audit"]
print(audit.to_string(index=False) if not audit.empty else "registry and dataset agree")

## 6. Baselines and honest evaluation

Walk-forward by season: train on every prior season, predict the next, never the reverse.

Read `brier_skill` first. Zero means the model has learnt nothing beyond the base rate;
negative means it is actively worse. `calibration_slope` near 1.0 means the probabilities
can be taken at face value — which matters if you plan to feed them into a season simulation.

In [ ]:
from src.models.train import walk_forward_evaluate, race_bootstrap

rows = []
for stage in ("pre_weekend", "post_quali"):
    for model in ("logistic", "gradient_boosting"):
        result = walk_forward_evaluate(dataset, stage=stage, model=model)
        if result.scores.empty:
            continue
        summary = result.summary()
        rows.append({"stage": stage, "model": model, "n_features": len(result.features),
                     **{k: round(float(summary[k]), 4) for k in
                        ("brier", "brier_skill", "roc_auc", "pr_auc", "calibration_slope")}})
comparison = pd.DataFrame(rows)
comparison

In [ ]:
best = walk_forward_evaluate(dataset, stage="post_quali", model="gradient_boosting")
print(best.scores[["season", "n", "observed_rate", "brier", "brier_skill",
                   "roc_auc", "pr_auc"]].round(4).to_string(index=False))
print()
# Rows within a race are correlated — one first-lap pile-up retires several cars — so the
# interval resamples whole races rather than rows.
interval = race_bootstrap(best.predictions, base_rate=float(dataset["dnf"].mean()))
print(f"brier_skill = {interval['mean']:+.4f}   95% CI "
      f"[{interval['ci_low']:+.4f}, {interval['ci_high']:+.4f}]")

In [ ]:
# Calibration: do the predicted probabilities mean what they say?
pred = best.predictions.copy()
pred["bucket"] = pd.qcut(pred["predicted"], 10, duplicates="drop")
curve = pred.groupby("bucket", observed=True).agg(
    predicted=("predicted", "mean"), observed=("dnf", "mean"), n=("dnf", "size"))

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, curve[["predicted", "observed"]].max().max()],
        [0, curve[["predicted", "observed"]].max().max()],
        "k--", lw=1, label="perfect calibration")
ax.scatter(curve["predicted"], curve["observed"], s=curve["n"] / 3)
ax.set_xlabel("predicted retirement probability")
ax.set_ylabel("observed retirement rate")
ax.set_title("Calibration by decile of predicted risk")
ax.legend()
plt.tight_layout()
curve.round(4)

## 7. Does measuring the circuit actually help?

The question this dataset was built to answer. Drop the whole track block and see what it
costs. `delta < 0` means removing the group hurt, so the group was carrying weight.

Group the features rather than dropping them one at a time: `track_speed_index` and
`speed_mean_kph` say nearly the same thing, so dropping either alone proves little.

In [ ]:
from src.models.train import ablate_features, track_feature_group

track = track_feature_group(dataset)
groups = {
    "track profile (measured circuit character)": track,
    "driver history": [c for c in dataset.columns if c.startswith("driver_")],
    "team history": [c for c in dataset.columns
                     if c.startswith(("team_", "teammate_"))],
    "circuit attrition history": [c for c in dataset.columns if c.startswith("circuit_")],
    "grid position": ["grid_position", "grid_position_pct", "is_back_half_of_grid"],
}
print(f"track feature group: {len(track)} columns")
ablation = ablate_features(dataset, groups, stage="post_quali", model="gradient_boosting")
ablation

In [ ]:
from src.models.train import permutation_importance_report

importance = permutation_importance_report(
    dataset, stage="post_quali", test_season=int(dataset["Year"].max()), n_repeats=10)
top = importance.head(20)

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(range(len(top)), top["brier_increase"], xerr=top["std"], color="steelblue")
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top["feature"], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("increase in Brier score when the feature is shuffled")
ax.set_title("Out-of-sample permutation importance")
plt.tight_layout()
top.round(5)

## 8. Where to take it next

In rough order of expected value:

1. **Model the two causes separately.** Mechanical failures and racing incidents are
   different processes with different drivers. `dnf_cause` is already in the dataset;
   fit one model per cause and add the probabilities, or fit a competing-risks model.
   This is the change most likely to beat the single binary model.
2. **Add a race-level shared shock.** Retirements cluster: a first-lap pile-up or a wet
   race takes out several cars at once. A model treating driver-races as independent will
   be systematically overconfident about the *number* of finishers, even when each
   individual probability is well calibrated. A hierarchical model with a per-race random
   effect fixes that, and matters a lot if these feed a season simulation.
3. **Feed it back into the points model.** A finishing probability is a natural input to
   notebook 2.0: expected points = P(finish) × E[points | finish]. That is likely the
   single most useful downstream use of this work.
4. **Test the two stress indices properly.** The ablation above is one test on limited
   data. `SPEED_INDEX_WEIGHTS`, `MECHANICAL_STRESS_WEIGHTS` and
   `INCIDENT_EXPOSURE_WEIGHTS` in `track_profile.py` are hand-chosen hypotheses; a
   supervised weighting (fit on training seasons only) may do better, at the cost of
   interpretability.
5. **Add tyre and stint data.** FastF1 exposes compound and stint length. Tyre failures
   and late-race incidents both relate to stint age, which nothing here currently sees.